<h2 style="color:#00bf00;">Exercise Week 2 Day 2 </h2> 
  <span style="color:#00bf00;">AI Orchestrating Agents with Code and LLMs using email: <br>
  <small><i>
   The business application is a Sales Development Representative (SDR) for an Autobiography Publishing company. <br>In this case, for a Story Development Researcher.<br><br>
   This exercise extends Parts 1-3 of the course to Parts 4-7:<br>
    - and switching to using email provider Resend <span style="color:#ff7800;">(See RESEND_INSTRUCTIONS.md)</span> <br>
    Part 4 another iteration orchestrating by code:<br> 
    <small>Here we add an **evaluator–optimizer loop**: a critic scores each draft and gives feedback, and a reviser rewrites it.</small><br>
    Part 5 another iteration orchestrating by LLM: <br>
    <small>Now the **Sales Manager LLM** decides the flow. It gets the writers, the critic and the reviser as **tools**, and runs the iteration itself.</small><br>
    Part 6 keeps the conversation going <br>
    Part 7 Resend handles polling and receiving email: <br>
    <small>To receive real replies, Resend needs an inbound webhook on a verified domain (`onboarding@resend.dev` can't receive mail)</small> <br>
    <br><small><i>Implemented at demo site,  
    <a href="https://lola-aurora.com/submit">https://lola-aurora.com</a>, with human-in-the-loop. <br>
    The switch is set where all email goes to EMAIL_ADDRESS with a "Review copy" banner.
    </span>

## Week 2 Day 2 - Orchestration

Our first Agentic Framework project!!

### Part 1: Email Setup

### Part 2: Orchestrating by code

### Part 3: Orchestrating by LLMs

- 3a: via Tools
- 3b: via Handoffs

## Part 1: Email Setup

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">IMPORTANT PLEASE READ - Sending Emails</h2>
            <span style="color:#ff7800;">We are going to write an agent that sends emails. The best way to do this is with an email provider
            like SendGrid or Resend. But this requires a fair amount of heavy-lifting: you need to send from a proper email host where you own the domain and you can prove it with DNS records. It's quite a hassle.<br/><br/>
            So we're going to do it the free & simple way: using SMTP server configuration to send direct from your email. It's an easy setup, but it's not as powerful (eg you can't receive emails). If you'd like to take this further, use SendGrid or Resend instead..<br/><br/>
            And actually the email send is optional. We're doing this to demonstrate an Agent sending emails. The important part is the Agent work, not the email sending. Feel free to swap out the function for a Pushover push if you'd prefer.
            </span>
        </td>
    </tr>
</table>


<h4 style="color:#008f00;">In this exercise, we will do this with email provider,  Resend</h4> 
    <span style="color:#ff7800;">See RESEND_INSTRUCTIONS.md</span> <br><br><br>

|  .env file                           |                         | 
| ------------------------------------ | ----------------------- |
| `RESEND_API_KEYS=re_`                | Resend api key          |  
| `AGENT_EMAIL=onboarding@resend.dev`  | without a domain, or    |    
| `AGENT_EMAIL=xxx@yourdomain.com`     | with a domain           |
| `EMAIL_ADDRESS=xxxx`                 | a working email address |  


## Setting up to send emails from Render 

### STEP 1: Sign up for a Render Account 

Add the default sandbox address to your .env file:  

`AGENT_EMAIL=onboarding@resend.dev`  

### STEP 2: Create a Resend API Key 

Add to your .env file:  

`RESEND_API_KEYS=re_`  

### STEP 3: Add in your email address:

`EMAIL_ADDRESS=xxx`

Remember to Save the .env file!

In [ ]:
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph
from openai.types.responses import ResponseTextDeltaEvent
import os
import asyncio
import resend  
load_dotenv(override=True)
MODEL_NAME = "gpt-5.4-mini"

RESEND_API_KEY = os.environ["RESEND_API_KEY"]
resend.api_key = RESEND_API_KEY
AGENT_EMAIL = os.getenv("AGENT_EMAIL")

In [ ]:
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")

if EMAIL_ADDRESS:
    print("Email address is set")
else:
    print("Email address is not set")

if AGENT_EMAIL:
    print("Agent Email address is set")
else:
    print("Agent Email address is not set")

USE_EMAIL = EMAIL_ADDRESS and AGENT_EMAIL 

if USE_EMAIL:
    print("Email is set up and we will try using it")
else:
    print("Email is not set up; we will send push notifications instead")

In [ ]:
def send_email(subject: str, text_body, html_body: str) -> str:
    """Tool for an AI agent to send an email notification."""
    response = resend.Emails.send({
        "from": AGENT_EMAIL,
        "to": [EMAIL_ADDRESS],
        "subject": subject,
        "html": html_body,
        "text": text_body
    })
    return f"Email sent successfully. ID: {response['id']}"

In [ ]:
send_email("Testing testing 123", "Fingers crossed..", "<html><body><strong>Fingers</strong> crossed..</body></html>")

### If this didn't work, then uncomment the below so that we don't use emails

<h5 style="color:#00bf00;">For this exersise, keep USE_EMAIL = False </h5> 

In [ ]:
#USE_EMAIL = False

### Our fallback strategy - send a push

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
def send_message(subject, text_body, html_body):
    if USE_EMAIL:
        send_email(subject, text_body, html_body)
    else:
        push(f"Subject: {subject}\n\n{text_body}")

### OK Now everything should work!

In [ ]:
send_message("Big news", "Communications are a go!", "<html><body>Communications are a <strong>go!</strong></body></html>")

## Agent Orchestration

There are 2 models for Agent Orchestration; by code and by LLMs.

By code: more predictable and deterministic.

By LLMs: more powerful.

An excellent write-up is here:

https://openai.github.io/openai-agents-python/multi_agent/

We will start with by Code.

## Part 2: Orchestrating by Code

In [ ]:
intro = """
You are a sales agent working for Autobiography Publishing, 
a company that provides a SaaS tool powered by AI, combining private autobiography publishing with lifetime digital archiving.
You write compelling sales emails that are likely to get a response.
"""

instructions1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instructions2 = intro + "Your email style is witty, engaging, and humorous."
instructions3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [ ]:
sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=MODEL_NAME)
sales_agent2 = Agent(name="Humorous Sales Agent", instructions=instructions2, model=MODEL_NAME)
sales_agent3 = Agent(name="Executive Sales Agent", instructions=instructions3, model=MODEL_NAME)

In [ ]:
result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

In [ ]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

In [ ]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""

sales_picker = Agent(name="Sales_picker", instructions=decision, model=MODEL_NAME)

In [ ]:
message = "Write a cold sales email"

with trace("Sales selection workflow"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")

Now go and check out the trace:

https://platform.openai.com/traces

### Now we will add a tool to the mix.

In [ ]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    send_message(subject, text_body, html_body)
    return "Email sent successfully"

### This has automatically been converted into a tool, with the boilerplate json created

In [ ]:
send_email_tool.params_json_schema

In [ ]:
decision = """
You pick the best cold sales email from the given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""

require_tool = ModelSettings(tool_choice="required")

sales_sender = Agent(name="Sales Sender", instructions=decision, model=MODEL_NAME, tools=[send_email_tool], model_settings=require_tool)

In [ ]:
message = "Write a cold sales email"

with trace("Sales selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    response = await Runner.run(sales_sender, emails)

    print(f"Final response:\n{response.final_output}")

### Did that work?!

See the traces for more! This is a great way to debug. Smaller models might require more time and experimentation.

https://platform.openai.com/traces

## Part 3: Orchestrating by LLMs

### 3a: via Tools

The simplest way to have 1 Agent choose to invoke another is by treating it as a tool call.

The OpenAI Agents SDK gives a very simple way to do this.

This works best when the flow is:

Agent A -> Agent B -> Agent A

And for the classic "Planning Agent" situation.

In [ ]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool1

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [ ]:
tool1 = sales_agent1.as_tool(tool_name="sales_email_writer_1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_email_writer_2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_email_writer_3", tool_description=description)

tools = [tool1, tool2, tool3, send_email_tool]

tools

## And now it's time for our Sales Manager - our planning agent

In [ ]:
instructions = """
You are a Sales Manager at Autobiography Publishing. Your goal is to find the single best cold sales email using the sales_writer tools.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use your tool to send the best email (and only the best email) to the user. Only send 1 email.
"""

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=MODEL_NAME)

### Optional: a diagram of our Sales Manager

The next cell draws a picture of the Sales Manager and its tools, using a free program called Graphviz. This step is optional! If you skip it, the two `draw_graph` cells will show an error. That's fine; just carry on with the lab.

To install Graphviz, open the Terminal in Cursor with Ctrl+backtick (`` ` ``) and run the command for your computer:

- Mac: `brew install graphviz` (you'll need Homebrew from https://brew.sh)
- Windows: `winget install graphviz`, then restart Cursor and run this notebook again from the top
- Linux: `sudo apt install graphviz`

In [ ]:
draw_graph(sales_manager)

In [ ]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

## Remember to check the trace

https://platform.openai.com/traces

And then check your email!! Also look in your Junk / Spam folder - after all, this is basically a spam message..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Unreliable results?</h2>
            <span style="color:#ff7800;">This is par for the course with Agentic AI, and particularly with Orchestrating by LLM.
            To resolve it, you'll need to experiment and iterate with the prompts. Particularly with smaller models, more experiments
            may be needed to get reliable outcomes.
            </span>
        </td>
    </tr>
</table>

## Part 3: Orchestrating by LLMs

### 3a: via Handoffs

I am not a fan of handoffs. They seem very unreliable. They're not used consistently by other frameworks.

Behind the scenes, OpenAI Agents SDK has implemented these with Tools anyway.

### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

A -> B -> A

With handoffs, control passes across

A -> B

In [ ]:

instructions = """
You are a Sales Manager at ComplAI. You get your sales team to draft emails, then send them all to a sales picker.
"""

task = """
Follow these steps:

1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.
 
2. Handoff to the sales sender to choose and send the best email.
"""

tools = [tool1, tool2, tool3]
handoffs = [sales_sender]

sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, handoffs=handoffs, model=MODEL_NAME)

In [ ]:
draw_graph(sales_manager)

In [ ]:
with trace("Sales manager"):
    result = await Runner.run(sales_manager, task)

### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

Note that handoffs can be unrealiable and a little bit frustrating. I needed to force the tool use otherwise this didn't work. If you don't get reliable behavior, try iterating on the prompts - or use a larger model. And enjoy the process; this is what Agentic AI is all about!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Try extending with some more orchestration - perhaps another iteration of the email to refine it. 
            Use Orchestrating via Code and Orchestrating via LLMs.  
            HARD CHALLENGE: switch to using a professional email provider like SendGrid or Resend. Then handle the user replying. 
            Then have the SDR respond to keep the conversation going!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../../../assets//business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>

<h2 style="color:#00bf00;">Parts 4 - 6: Another Iteration of the Email</h2> 
    <span style="color:#00bf00;">Suggested Order: <br>

1. Part 4, small first. <br>
-Start with the cell that runs the critic on one draft. <br>
-Check that it ends with SCORE: n/10 and that the parsed score isn’t 0. <br>
-Then run the full workflow with max_rounds=1 before trying 2. <br>
-A full run makes roughly 10–18 model calls and sends one email.<br><br>
2. Part 5. Run it once and open the trace.<br> 
-Check that the manager called all three writers, the critic and the reviser before the handoff, and that exactly one email arrived. <br>
-LLM-run flows vary from run to run, so a second run is worth doing.<br><br>
3. Part 6.<br>
-Run the paste-a-reply cell first. It sends one real reply to you.<br>
-Then run the simulated conversation, which is set to send=False (nothing is sent).<br>
-Then run the SDR Manager cell, which sends.<br>
-Try one reply like “please remove me from your list” to check that nothing is sent back.
</span>

## Part 4: Another Iteration of the Email — Orchestrating by Code

So far, each draft is written once and the picker chooses among first drafts.
Here we add an **evaluator–optimizer loop**: a critic scores each draft and gives feedback,
and a reviser rewrites it. The loop runs until the score reaches a target or we hit a round limit.
**Our Python code** controls the flow: what runs, in what order, and when to stop.

```
3 writers (parallel) → [critic → reviser]×N per draft (parallel) → picker → subject + HTML (parallel) → send
```

### Part 4a: Test, small first. 
1. Start with the cell that runs the critic on one draft.

- *Check that it ends with SCORE: n/10 and that the parsed score isn’t 0.*

2. Then run the full workflow with max_rounds=1 before trying 2. 

- The workflow cell calls it without setting max_rounds, so it uses the default of 2: 

        max_rounds: int = 2, target_score: int = 8) -> str:

- To test with one round first, change that line in the workflow cell to:

        max_rounds=1, int = 2, target_score: int = 8) -> str:
    
3. A full run makes roughly 10–18 model calls and sends one email.

In [ ]:
critic_instructions = """
You are a demanding Sales Director at Autobiography Publishing reviewing a cold sales email.
Give 3-5 short, concrete, actionable improvements, covering: the subject line hook, personalization,
clarity of the value proposition, the call to action, and length.
Do not rewrite the email; give the critique only.
On the final line, write exactly: SCORE: n/10
"""

reviser_instructions = intro + """
You revise a cold sales email based on a critique from your Sales Director.
Keep the email's original style and voice, apply every point of the feedback,
and reply with the revised email only, with no commentary.
"""

sales_critic = Agent(name="Sales Critic", instructions=critic_instructions, model=MODEL_NAME)
sales_reviser = Agent(name="Sales Reviser", instructions=reviser_instructions, model=MODEL_NAME)

Test 1: *Run the full workflow, change 'max_rounds:' with 'max_rounds=1' before trying 2.*

In [ ]:
import re

def parse_score(critique: str) -> int:
    """Pull the n out of 'SCORE: n/10'; return 0 if the critic forgot it."""
    match = re.search(r"SCORE:\s*(\d+)", critique)
    return int(match.group(1)) if match else 0


async def refine_email(name: str, draft: str, max_rounds: int = 2, target_score: int = 8) -> str:
    """Critique and revise one draft until it scores >= target_score or max_rounds is reached."""
    for round_number in range(1, max_rounds + 1):
        critique = (await Runner.run(sales_critic, draft)).final_output
        score = parse_score(critique)
        print(f"{name} | round {round_number} | score {score}/10")
        if score >= target_score:
            break
        revise_prompt = f"Original email:\n\n{draft}\n\nCritique:\n\n{critique}\n\nRevise the email."
        draft = (await Runner.run(sales_reviser, revise_prompt)).final_output
    return draft

Before refining, a quick look at what the critic does with one draft:

In [ ]:
draft = (await Runner.run(sales_agent3, "Write a cold sales email")).final_output
critique = (await Runner.run(sales_critic, draft)).final_output
print(critique)
print("\nParsed score:", parse_score(critique))

### Formatting agents: subject line and HTML

Instead of the writers' plain text, two small agents prepare the email for sending.
They don't depend on each other, so code runs them in parallel.

Test 2: *Run the full workflow with 'max_rounds', instead of 'max_rounds=1'.*

* max_rounds is a setting of the refine_email function in Part 4. It sets how many critique-and-revise rounds each draft can go through.
* The workflow cell calls it without setting max_rounds, so it uses the default of 2: refine_email(agent.name, draft)
* To test with one round first, change that line in the workflow cell to: refine_email(agent.name, draft, max_rounds=1)

In [ ]:
subject_instructions = """
You write a subject line for a cold sales email.
You are given the email body. Write a subject line that is likely to get it opened.
Reply with the subject line only.
"""

html_instructions = """
You convert a plain-text or markdown email body into a clean, simple HTML email
with a clear layout. Reply with the HTML only — no code fences, no commentary.
"""

subject_writer = Agent(name="Email Subject Writer", instructions=subject_instructions, model=MODEL_NAME)
html_converter = Agent(name="HTML Email Body Converter", instructions=html_instructions, model=MODEL_NAME)


async def format_and_send(body: str) -> str:
    """Write a subject and HTML body in parallel, then send through send_message."""
    subject_result, html_result = await asyncio.gather(
        Runner.run(subject_writer, body),
        Runner.run(html_converter, body),
    )
    subject = subject_result.final_output.strip()
    html_body = html_result.final_output
    send_message(subject, body, html_body)
    return subject

### The complete code-orchestrated workflow

In [ ]:
message = "Write a cold sales email"
writers = [sales_agent1, sales_agent2, sales_agent3]

with trace("Iterated sales workflow (code)"):
    # 1. First drafts, in parallel
    results = await asyncio.gather(*[Runner.run(agent, message) for agent in writers])
    drafts = [result.final_output for result in results]

    # 2. Another iteration: critique and revise every draft, in parallel
    refined = await asyncio.gather(
        *[refine_email(agent.name, draft) for agent, draft in zip(writers, drafts)]
    )

    # 3. Pick the best refined email
    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(refined)
    best = (await Runner.run(sales_picker, emails)).final_output

    # 4. Subject + HTML, then send
    subject = await format_and_send(best)

print(f"Sent: {subject}\n\n{best}")

### Part 4b: Results

What one round is: the critic scores the draft. If the score is 8 or higher, the draft is done. If not, the reviser rewrites it.

|	          |   Per draft (up to)	    | Whole run, 3 draft (upto)                           |
| ----------- | ----------------------- | --------------------------------------------------- |
|max_rounds=1 | 1 critique + 1 rewrite	| 3 writers + 6 + picker + subject + HTML = 12 calls  |
|max_rounds=2 | 2 critiques + 2 rewrite | 3 writers + 12 + picker + subject + HTML = 18 calls |

If a draft scores 8 or more, it stops early and uses fewer calls.

>Why start with 1: it’s quicker and cheaper, and it shows whether the loop itself works. In the printed lines you’ll see the scores the critic gave and whether a rewrite happened. Once that works, going back to 2 adds the real “another round”: the rewritten draft gets scored again.

>One thing you’ll notice: the last rewrite is never scored. With max_rounds=1, the printout shows the first draft’s score, but not the score of its rewrite.

https://platform.openai.com/traces

## Part 5: Another Iteration of the Email — Orchestrating by LLMs

Now the **Sales Manager LLM** decides the flow. It gets the writers, the critic and the reviser as **tools**, and runs the iteration itself. When it has the winner, it uses a **handoff** to pass control
to an Email Manager agent, which formats and sends the email.

- **Tool**: the manager calls an agent, gets the result back, and stays in charge.
- **Handoff**: the manager passes the conversation on; the Email Manager finishes the job.

##### Testing Suggested Order:
* Run it once and open the trace. 
* Check that the manager called all three writers, the critic and the reviser before the handoff, and that exactly one email arrived. 
* LLM-run flows vary from run to run, so a second run is worth doing.

In [ ]:
critic_tool = sales_critic.as_tool(
    tool_name="sales_email_critic",
    tool_description="Critique one cold sales email. Input: the full email. Returns feedback ending with 'SCORE: n/10'.",
)

reviser_tool = sales_reviser.as_tool(
    tool_name="sales_email_reviser",
    tool_description=(
        "Revise one cold sales email. Input must contain BOTH the full original email "
        "and the full critique for it. Returns the revised email."
    ),
)

subject_tool = subject_writer.as_tool(
    tool_name="subject_writer",
    tool_description="Write a subject line for an email. Input: the email body.",
)

html_tool = html_converter.as_tool(
    tool_name="html_converter",
    tool_description="Convert a plain-text email body to HTML. Input: the email body.",
)

In [ ]:
emailer_instructions = """
You are an email formatter and sender. You receive the body of one email to be sent.
1. Use the subject_writer tool to write a subject line for the email.
2. Use the html_converter tool to convert the body to HTML.
3. Use the send_email_tool exactly once, passing the subject, the original plain-text body as text_body,
   and the HTML as html_body.
"""

emailer_agent = Agent(
    name="Email Manager",
    instructions=emailer_instructions,
    tools=[subject_tool, html_tool, send_email_tool],
    model=MODEL_NAME,
    handoff_description="Format the final email (subject line and HTML) and send it",
)

In [ ]:
manager_v2_instructions = """
You are a Sales Manager at Autobiography Publishing. Your goal is to send the single best cold sales email,
improved through critique and revision.

Follow these steps carefully:
1. Generate drafts: use all three sales_email_writer tools. Just instruct each to write a sales email.
   Do not proceed until you have one draft from each tool.
2. Critique: send each draft to the sales_email_critic tool.
3. Revise (another iteration): for each draft, call the sales_email_reviser tool with the full draft AND its full critique.
4. Optional final polish: pick the strongest revised email. If its critic score was below 8,
   run ONE more critique and revision on that email only.
5. Handoff: hand off ONLY the winning email body to the Email Manager agent, which formats and sends it.

Crucial rules:
- Always use the tools to write, critique and revise; never write or edit an email yourself.
- Hand off exactly one email. Do not send email yourself.
"""

manager_v2_tools = [tool1, tool2, tool3, critic_tool, reviser_tool]

sales_manager_v2 = Agent(
    name="Sales Manager v2",
    instructions=manager_v2_instructions,
    tools=manager_v2_tools,
    handoffs=[emailer_agent],
    model=MODEL_NAME,
)

In [ ]:
draw_graph(sales_manager_v2)

In [ ]:
task_v2 = "Send out a cold sales email addressed to Dear CEO from Alice"

# 3 writers + 3 critiques + 3 revisions + optional polish + handoff exceeds the default max_turns of 10,
# so we raise it. Parallel tool calls in one turn usually keep the real count lower.
with trace("Iterated sales workflow (LLM)"):
    result = await Runner.run(sales_manager_v2, task_v2, max_turns=30)

print(f"Finished with agent: {result.last_agent.name}\n")
print(result.final_output)

https://platform.openai.com/traces

In the trace, compare Part 4 and Part 5:
- **Part 4**: the number of rounds is fixed by `max_rounds` and `target_score`. The flow is predictable, cheaper, and easy to test.
- **Part 5**: the manager decides how many calls to make and when to stop. It's more flexible, but the cost and order vary from run to run, so check that it really called all three writers, the critic and the reviser before handing off.

## Part 6: Handling the Prospect's Reply — and Keeping the Conversation Going

A real SDR doesn't stop after the first email. Here, when the prospect replies:
1. A **Reply Classifier** works out what the prospect wants (interested, question, objection, not now, unsubscribe).
2. The **SDR** writes a response that fits the intent and the whole thread, aiming to move the prospect toward a short call.
3. The response is sent, and the thread grows with each turn.

**Where do replies come from?** To receive real replies, Resend needs an inbound webhook on a verified domain
(`onboarding@resend.dev` can't receive mail). Until that's set up, you paste the reply into a cell,
or let a **Simulated Prospect** agent play the customer.

The thread is a plain Python list of dicts, so every agent sees the full conversation history.

In [ ]:
# Start the thread with the email sent in Part 4 (falls back to a placeholder if Part 4 wasn't run)
first_email = best if "best" in globals() else "Hi, I'm Alice from Autobiography Publishing..."

thread = [{"role": "SDR", "text": first_email}]

def format_thread(thread: list) -> str:
    """Turn the thread into a readable transcript for the agents."""
    return "\n\n".join(f"--- {entry['role']} ---\n{entry['text']}" for entry in thread)

### The agents

In [ ]:
classifier_instructions = """
You classify a prospect's latest reply in a sales email thread for Autobiography Publishing.
Reply with exactly one label and nothing else:
INTERESTED, QUESTION, OBJECTION, NOT_NOW, UNSUBSCRIBE
"""

sdr_instructions = intro + """
You are now in an ongoing email conversation with a prospect who replied to your cold email.
Read the whole thread, then write the next reply from the SDR.
- Answer exactly what the prospect asked; don't invent prices, features, or guarantees you don't know.
  If you don't know something, say you'll confirm it.
- INTERESTED or QUESTION: answer, then propose a 15-minute call with two specific time options.
- OBJECTION: acknowledge it honestly, address it briefly, and offer a low-commitment next step.
- NOT_NOW: be gracious and ask when it would be good to follow up.
Keep it short (under 150 words), warm and human. Reply with the email body only.
"""

prospect_instructions = """
You play a busy prospect who received a cold sales email from Autobiography Publishing,
an AI-powered private autobiography publishing and lifetime digital archiving service.
You are a 62-year-old retired executive, mildly curious but skeptical about privacy and cost.
Read the thread and write your next reply as the prospect: short, realistic, sometimes asking a question
or raising an objection. If a call has been agreed with a specific time, say goodbye politely.
Reply with the email body only.
"""

reply_classifier = Agent(name="Reply Classifier", instructions=classifier_instructions, model=MODEL_NAME)
sdr_responder = Agent(name="SDR Responder", instructions=sdr_instructions, model=MODEL_NAME)
simulated_prospect = Agent(name="Simulated Prospect", instructions=prospect_instructions, model=MODEL_NAME)

### Orchestrating by Code: one reply → one response

The code decides the order: classify, then branch. An unsubscribe request stops the SDR from writing back.

In [ ]:
async def handle_reply(thread: list, reply_text: str, send: bool = True) -> str:
    """Add the prospect's reply to the thread, classify it, write and send the SDR response."""
    thread.append({"role": "Prospect", "text": reply_text})
    transcript = format_thread(thread)

    intent = (await Runner.run(reply_classifier, transcript)).final_output.strip().upper()
    print(f"Intent: {intent}")

    if intent == "UNSUBSCRIBE":
        print("Prospect asked to stop. No reply sent; remove them from the list.")
        return intent

    sdr_input = f"Prospect intent: {intent}\n\nThread so far:\n\n{transcript}"
    response = (await Runner.run(sdr_responder, sdr_input)).final_output
    thread.append({"role": "SDR", "text": response})

    if send:
        html_body = (await Runner.run(html_converter, response)).final_output
        send_message("Re: Your story, preserved", response, html_body)
    return intent

In [ ]:
# Paste a real reply here, or edit this one
prospect_reply = "This sounds interesting, but who can see my stories? And roughly what does it cost?"

with trace("Handle prospect reply (code)"):
    await handle_reply(thread, prospect_reply)

print(thread[-1]["text"])

### Keeping the conversation going: a simulated back-and-forth

The Simulated Prospect replies, the SDR answers, and the loop repeats until the prospect
unsubscribes or we hit `max_turns`. `send=False` keeps your inbox quiet while you test;
set it to `True` to send every SDR reply.

In [ ]:
async def run_conversation(thread: list, max_turns: int = 3, send: bool = True) -> list:
    for turn in range(1, max_turns + 1):
        prospect_text = (await Runner.run(simulated_prospect, format_thread(thread))).final_output
        print(f"\n===== Turn {turn} =====\nPROSPECT:\n{prospect_text}\n")
        intent = await handle_reply(thread, prospect_text, send=send)
        if intent == "UNSUBSCRIBE":
            break
        print(f"SDR:\n{thread[-1]['text']}")
    return thread


with trace("Simulated SDR conversation (code)"):
    await run_conversation(thread, max_turns=3, send=False)

### Orchestrating by LLMs: the SDR Manager decides how to respond

Here a single **SDR Manager** agent gets the thread and decides what to do with its tools:
classify the reply, draft a response, ask the critic for a review if the response is important (for example, an objection),
revise it, and send it. Code only supplies the thread and records the result.

In [ ]:
classifier_tool = reply_classifier.as_tool(
    tool_name="reply_classifier",
    tool_description="Classify the prospect's latest reply. Input: the full thread. Returns one intent label.",
)

responder_tool = sdr_responder.as_tool(
    tool_name="sdr_responder",
    tool_description="Draft the SDR's next reply. Input: the prospect intent label plus the full thread.",
)

sdr_manager_instructions = """
You are the SDR Manager at Autobiography Publishing, handling an ongoing email thread with a prospect.
Follow these steps:
1. Use reply_classifier on the full thread to get the prospect's intent.
2. If the intent is UNSUBSCRIBE: do not send anything. Reply with the single word UNSUBSCRIBE.
3. Otherwise, use sdr_responder with the intent and the full thread to draft a reply.
4. If the intent is OBJECTION or QUESTION, use sales_email_critic on the draft, then sales_email_reviser
   with the draft AND its critique, for one more iteration.
5. Use send_email_tool exactly once with a subject starting with "Re: ", the reply as text_body,
   and a simple HTML version as html_body.
6. Finally, reply with the exact email body you sent and nothing else.
Never write the email yourself; always use the tools.
"""

sdr_manager = Agent(
    name="SDR Manager",
    instructions=sdr_manager_instructions,
    tools=[classifier_tool, responder_tool, critic_tool, reviser_tool, send_email_tool],
    model=MODEL_NAME,
)

In [ ]:
prospect_reply = "Honestly I'm not sure I need this. My kids can just keep my old photos."
# prospect_reply = "Please remove me from your list."

thread.append({"role": "Prospect", "text": prospect_reply})

with trace("Handle prospect reply (LLM)"):
    result = await Runner.run(sdr_manager, format_thread(thread), max_turns=20)

if result.final_output.strip() != "UNSUBSCRIBE":
    thread.append({"role": "SDR", "text": result.final_output})

print(format_thread(thread))

In [ ]:
https://platform.openai.com/traces

<h2 style="color:#00bf00;">Next step for real replies:</h2> 
    <span style="color:#ff7800;">`onboarding@resend.dev`can't receive mail<br></span> 
    <span style="color:#00bf00;">
    - Add a Resend inbound webhook on your domain (or poll a mailbox), and <br>
    - call `handle_reply(thread, incoming_text)` from it. <br>
    - Keep one `thread` per prospect, for example in a dict keyed by email address. <br>
</span>



**Next step for real replies:** add a Resend inbound webhook on your domain (or poll a mailbox),
and call `handle_reply(thread, incoming_text)` from it. Keep one `thread` per prospect, for example in a dict keyed by email address.

In [ ]:
# To confirm your environment really has resend installed
import resend
print(resend.__version__, hasattr(resend.Emails, "Receiving"))

In [ ]:
# See the whole conversation with you so far
print(format_thread(state["threads"].get(EMAIL_ADDRESS.lower(), [])))

In [ ]:
# Quick test before Part 7: run this in a cell. If it prints ready, Part 7 has everything it needs
for name in ["resend", "EMAIL_ADDRESS", "AGENT_EMAIL", "intro", "html_converter",
             "format_thread", "reply_classifier", "sdr_responder"]:
    if name not in globals():
        print("missing:", name)
print("ready")

## Part 7: Real Replies — Polling the Resend Inbox

Resend stores every email that arrives at a receiving-enabled domain, even with no webhook set up,
and the **Receiving API** lets us list and read them. So the notebook can **poll**: every minute it asks
Resend "anything new?", and runs each new reply through the classifier and the SDR.

```
Prospect replies → MX record → Resend inbox → (notebook polls) → classify → SDR reply → send in the same thread
```

**One-time setup (outside the notebook)**
1. Resend → **Domains** → your subdomain → turn on **Receiving**, and copy the MX record it shows.
2. Cloudflare → DNS → add that **MX** record on the same subdomain as `AGENT_EMAIL` (priority from Resend, usually 10).
3. Reply to one of your test emails, then check Resend → **Emails → Receiving**: the reply should be listed.
4. Update the SDK if resend is older than 2.55: `uv pip install -U resend` (Receiving needs a recent version).

**Safety rules built in**
- Replies are only processed from `ALLOWED_SENDERS` (just you, for now).
- Mail from `AGENT_EMAIL` itself is ignored, as are auto-replies, so the SDR can't loop with itself.
- Each received email is handled once; state is saved to `sdr_state.json`, so restarting the kernel doesn't send duplicates.
- `SEND_REPLIES = False` runs everything except the actual send.

In [ ]:
import json
import re
import time
from pathlib import Path
from email.utils import parseaddr

STATE_FILE = Path("sdr_state.json")
SEND_REPLIES = True         # set to True once the dry run looks right
ALLOWED_SENDERS = {EMAIL_ADDRESS.lower()}   # add real prospects here later
AGENT_ADDRESS = parseaddr(AGENT_EMAIL)[1].lower()


def load_state() -> dict:
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return {"processed_ids": [], "unsubscribed": [], "threads": {}}


def save_state(state: dict) -> None:
    STATE_FILE.write_text(json.dumps(state, indent=2))


state = load_state()

# Seed your own thread with the first email sent (from Part 4), if it isn't there yet
if EMAIL_ADDRESS.lower() not in state["threads"] and "best" in globals():
    state["threads"][EMAIL_ADDRESS.lower()] = [{"role": "SDR", "text": best}]
    save_state(state)

print(f"Threads: {list(state['threads'])} | processed: {len(state['processed_ids'])}")

### Reading a reply

A reply usually contains the whole earlier email quoted underneath (`On ... wrote:` and `>` lines).
We keep only the new text, so the agents don't re-read the thread twice.

In [ ]:
def strip_quoted(text: str) -> str:
    """Keep only the new part of a reply."""
    text = re.split(r"\n\s*On .{0,300}?wrote:", text, maxsplit=1, flags=re.DOTALL)[0]
    text = text.split("-----Original Message-----")[0]
    lines = [line for line in text.splitlines() if not line.lstrip().startswith(">")]
    return "\n".join(lines).strip()


def reply_body(email: dict) -> str:
    """Plain text of a received email, falling back to the HTML with tags removed."""
    text = email.get("text") or re.sub(r"<[^>]+>", " ", email.get("html") or "")
    return strip_quoted(text)


def is_auto_reply(email: dict) -> bool:
    headers = {key.lower(): value for key, value in (email.get("headers") or {}).items()}
    return headers.get("auto-submitted", "no").lower() != "no" or "x-autoreply" in headers


def fetch_new_emails(state: dict) -> list:
    """Return unprocessed received emails, oldest first, with full content."""
    listing = resend.Emails.Receiving.list({"limit": 20})
    new_items = [item for item in listing["data"] if item["id"] not in state["processed_ids"]]
    return [resend.Emails.Receiving.get(item["id"]) for item in reversed(new_items)]

In [ ]:
# Look before acting: what's in the inbox right now? (read-only)
for email in fetch_new_emails(state):
    print(email["created_at"], "|", email["from"], "|", email["subject"])
    print(reply_body(email)[:300], "\n")

### Replying in the same thread

`In-Reply-To` and `References` point at the prospect's message, and the subject starts with `Re:`,
so the response shows up in the same conversation in their email app, instead of as a new email.

In [ ]:
def send_reply(to: str, subject: str, text_body: str, html_body: str, message_id: str) -> str:
    subject = subject if subject.lower().startswith("re:") else f"Re: {subject}"
    params = {
        "from": AGENT_EMAIL,
        "to": [to],
        "subject": subject,
        "text": text_body,
        "html": html_body,
    }
    if message_id:
        message_id = message_id if message_id.startswith("<") else f"<{message_id}>"
        params["headers"] = {"In-Reply-To": message_id, "References": message_id}
    return resend.Emails.send(params)["id"]


async def process_email(email: dict, state: dict) -> None:
    """Classify one received email, write the SDR response, and send it (or dry-run)."""
    sender = parseaddr(email["from"])[1].lower()
    state["processed_ids"].append(email["id"])   # mark first, so a crash never causes a double send

    if sender == AGENT_ADDRESS or is_auto_reply(email):
        print(f"Skipped (own mail or auto-reply): {sender}")
    elif sender not in ALLOWED_SENDERS:
        print(f"Skipped (not an allowed sender): {sender}")
    elif sender in state["unsubscribed"]:
        print(f"Skipped (unsubscribed): {sender}")
    else:
        thread = state["threads"].setdefault(sender, [])
        thread.append({"role": "Prospect", "text": reply_body(email)})
        transcript = format_thread(thread)

        intent = (await Runner.run(reply_classifier, transcript)).final_output.strip().upper()
        print(f"{sender} | intent: {intent}")

        if intent == "UNSUBSCRIBE":
            state["unsubscribed"].append(sender)
        else:
            sdr_input = f"Prospect intent: {intent}\n\nThread so far:\n\n{transcript}"
            response = (await Runner.run(sdr_responder, sdr_input)).final_output
            html_body = (await Runner.run(html_converter, response)).final_output
            thread.append({"role": "SDR", "text": response})
            if SEND_REPLIES:
                sent_id = send_reply(sender, email["subject"], response, html_body, email.get("message_id"))
                print(f"Sent reply {sent_id}")
            else:
                print(f"DRY RUN — would send:\n{response}\n")

    save_state(state)

### Check once, then watch

Run **Check once** first (with `SEND_REPLIES = False`). When the drafted reply looks right,
set `SEND_REPLIES = True` and run **Watch**. Stop it any time with the notebook's stop button;
the state file means nothing is handled twice.

In [ ]:
# Check once
with trace("SDR inbox check"):
    for email in fetch_new_emails(state):
        await process_email(email, state)

In [ ]:
# Watch: poll every minute; create a file named STOP to end it cleanly
STOP_FILE = Path("STOP")

async def watch_inbox(state: dict, minutes: int = 15, every_seconds: int = 60) -> None:
    STOP_FILE.unlink(missing_ok=True)          # clear any old stop signal
    end_time = time.time() + minutes * 60
    while time.time() < end_time and not STOP_FILE.exists():
        new_emails = fetch_new_emails(state)
        if new_emails:
            with trace("SDR inbox watch"):
                for email in new_emails:
                    await process_email(email, state)
        else:
            print(time.strftime("%H:%M:%S"), "no new replies")
        # sleep in short steps so STOP is noticed quickly
        for _ in range(every_seconds // 5):
            if STOP_FILE.exists():
                break
            await asyncio.sleep(5)
    print("Watch stopped.")


await watch_inbox(state, minutes=15)

In [ ]:
# See the whole conversation with you so far
print(format_thread(state["threads"].get(EMAIL_ADDRESS.lower(), [])))

In [ ]:
# Watch: poll every minute for a while
async def watch_inbox(state: dict, minutes: int = 15, every_seconds: int = 60) -> None:
    end_time = time.time() + minutes * 60
    while time.time() < end_time:
        new_emails = fetch_new_emails(state)
        if new_emails:
            with trace("SDR inbox watch"):
                for email in new_emails:
                    await process_email(email, state)
        else:
            print(time.strftime("%H:%M:%S"), "no new replies")
        await asyncio.sleep(every_seconds)


await watch_inbox(state, minutes=15)

In [ ]:
# See the whole conversation with you so far
print(format_thread(state["threads"].get(EMAIL_ADDRESS.lower(), [])))